In [1]:
import pandas as pd
import xlrd

## APIPred

In [ ]:
apipred_df = pd.read_excel('../data/row/INPUT-APTAMER-PROTEIN-PAIRS.xls', header=1)
apipred_df = apipred_df[apipred_df['Class label'] == 'positive']
apipred_df.head(3)

,ID,Class label,DNA/RNA sequence,Target_ID,Protein sequence
0,1,positive,UCGGAGGUGGUUCAGCUCUGCAUCGACAGUUGGC,dsRNA_activated_protein_kinase,MDSTNYISKLFEYAQRQGQISDIKFEEVGTDGPDHLKTFTLRVVIK...
1,2,positive,CGGGGTGTTGTCCTGTGCTCTCCGGAGAGCACAGGACAACACCCCG,Human_Nuclear_Factor_kB_RelA_p65,MDELFPLIFPAEPAQASGPYVEIIEQPKQRGMRFRYKCEGRSAGSI...
2,3,positive,GCTGCAGTTGCACTGAATTCGCCTCTCGCCTCCGTACACTTAGTCG...,atSPL14,MDEVGAQVAAPMFIHQSLGRKRDLYYPMSNRLVQSQPQRRDEWNSK...


In [3]:
unique_sequences = apipred_df['DNA/RNA sequence'].drop_duplicates().reset_index(drop=True)
id_map = {
    seq: f"APIPred_{i+1}" for i, seq in enumerate(unique_sequences)
}
apipred_df['apt_name'] = apipred_df["DNA/RNA sequence"].map(id_map)
apipred_df.rename(columns={'Protein sequence': 'target_seq', 'DNA/RNA sequence': 'apt_seq', 'Target_ID': 'target_name'}, inplace=True)
apipred_df.drop(columns=['Class label', 'ID'], inplace=True)

apipred_df.drop_duplicates(inplace=True)
apipred_df['target_type'] = 'protein'
apipred_df['Database'] = 'APIPred'

## RSAPred

In [ ]:
rsa_df = pd.read_csv('../data/row/Aptamers_dataset_v1.csv', sep='\t')
rsa_df

,Entry_ID,SMILES,Target_RNA_sequence,Molecule_name,Molecule_ID,Target_RNA_name,Target_RNA_ID,pKd
0,39,C1=CC=C2C(=C1)C(=O)C3=C(C2=O)C(=C(C=C3NC4=CC(=...,GGGAGAATTCCCGCGGCAGAAGCCCACCTGGCTTTGAACTCTATGT...,Cibacron blue,Target_lig_27,RNA_APTAMER_CB_42,Target_4,4.000000
1,40,C1=CC=C2C(=C1)C(=O)C3=C(C2=O)C(=C(C=C3NC4=CC(=...,GGGAGAAUUCCCGCGGCGUUGGCCCAGGAUAAUAGGACGAAAUCCG...,Reactive Blue 4,Target_lig_28,RNA_APTAMER_B4_25,Target_5,3.221849
2,91,C1=NC(=C2C(=N1)N(C=N2)C3C(C(C(O3)COP(=O)(O)OP(...,GGGAAGGGAAGAAACUGCGGCUUCGGCCGGCUUCCC,ATP,Target_lig_65,RNA_Aptamer,Target_6,5.397940
3,92,C1=NC(=C2C(=N1)N(C=N2)C3C(C(C(O3)COP(=O)(O)O)O...,GGGAAGGGAAGAAACUGCGGCUUCGGCCGGCUUCCC,AMP,Target_lig_66,RNA_Aptamer,Target_6,8.301026
4,93,CC1=CC2=C(C=C1C)N(C3=NC(=O)NC(=O)C3=N2)CC(C(C(...,GGCGUGUAGGAUAUGCUUCGGCAGAAGGACACGCC,FAD,Target_lig_67,35 nucleotide RNA,Target_26,4.638272
...,...,...,...,...,...,...,...,...
515,2497,O=C(Cc1ccccc1)OC[C@H]1OC(=O)NC1CN1CCN(CC1)c1cc...,GAGGGUGGAACCGCGCUUCGGCGUCCCUC,"Compound (4R,5S)-1",Target_lig_1239,Antiterminator model RNA AM1A,Target_147,4.920819
516,2498,O=C(Cc1ccccc1)OCC1OC(=O)N[C@H]1CN1CCN(CC1)c1cc...,GAGGGUGGAACCGCGCUUCGGCGUCCCUC,"Compound (4S,5R)-1",Target_lig_1240,Antiterminator model RNA AM1A,Target_147,4.795880
517,2499,O=C(Nc1ccc(cc1)C(=O)C)OC[C@H]1OC(=O)NC1CN1CCN(...,GAGGGUGGAACCGCGCUUCGGCGUCCCUC,"Compound (4R,5S)-2",Target_lig_1241,Antiterminator model RNA AM1A,Target_147,5.522879
518,2500,O=C(Nc1ccc(cc1)C(=O)C)OCC1OC(=O)N[C@H]1CN1CCN(...,GAGGGUGGAACCGCGCUUCGGCGUCCCUC,"Compound (4S,5R)-2",Target_lig_1242,Antiterminator model RNA AM1A,Target_147,5.795880


In [5]:
rsa_df.rename(columns={'Target_RNA_sequence': 'apt_seq', 'Target_RNA_name': 'apt_name', 'Molecule_name': 'target_name', 'SMILES': 'target_seq'}, inplace=True)
rsa_df['target_type'] = 'small_molecule'

## PDBBind

In [ ]:
bind_df = pd.read_csv('../data/row/df_PDBBind_processed.csv', index_col=0)
bind_df.head(3)

,pdb,res,year,neglog_aff,neglog_aff_nor,affinity,ref,ligand,measurement,content,smiles,annotation,ligand_name
0,209d,3,1995,5.000000,0.294837,10uM,209d.pdf,PXA,IC50,GAAGCTTC,Cc1c2oc3c(C)cnc(C(=O)O)c3nc-2c(C(=O)O)c(N)c1=O,"['Structural, physical and biological characte...",NaN
1,1arj,NMR,1996,3.000000,0.058150,1mM,1arj.pdf,ARG,Kd,GGCAGAUCUGAGCCUGGGAGCUCUCUGCC,NC(=[NH2+])NCCC[C@H](N)C(=O)O,"['ARG-BOUND TAR RNA, NMR']",[(4S)-4-amino-4-carboxybutyl]-(diaminomethylid...
2,316d,3,1997,7.110138,0.544558,77.6nM,316d.pdf,PXF,IC50,GAAGCTTC,Cc1c2oc3c(C)cc(F)c(C(=O)O)c3nc-2c(C(=O)O)c(N)c1=O,['Selectivity of F8-actinomycin D for RNA:DNA ...,NaN


In [7]:
#If 'ligand_name' is empty, take the value from 'ligand'
bind_df['ligand_name'] = bind_df['ligand_name'].fillna(bind_df['ligand'])

In [8]:
# There is no aptamer name in the database, so we assign the name 'PDBBind_id'
# Creating a map of unique sequences and names
unique_sequences = bind_df['content'].drop_duplicates().reset_index(drop=True)
id_map = {
    seq: f"PDBBind_{i+1}" for i, seq in enumerate(unique_sequences)
}
bind_df['apt_name'] = bind_df['content'].map(id_map)
bind_df.rename(columns={'content': 'apt_seq', 'ligand_name': 'target_name', 'smiles': 'target_seq'}, inplace=True)


bind_df.drop_duplicates(subset=['apt_seq', 'target_seq', 'target_name', 'apt_name'], inplace=True)
bind_df['target_type'] = 'small_molecule'
bind_df.head(3)

,pdb,res,year,neglog_aff,neglog_aff_nor,affinity,ref,ligand,measurement,apt_seq,target_seq,annotation,target_name,apt_name,target_type
0,209d,3,1995,5.000000,0.294837,10uM,209d.pdf,PXA,IC50,GAAGCTTC,Cc1c2oc3c(C)cnc(C(=O)O)c3nc-2c(C(=O)O)c(N)c1=O,"['Structural, physical and biological characte...",PXA,PDBBind_1,small_molecule
1,1arj,NMR,1996,3.000000,0.058150,1mM,1arj.pdf,ARG,Kd,GGCAGAUCUGAGCCUGGGAGCUCUCUGCC,NC(=[NH2+])NCCC[C@H](N)C(=O)O,"['ARG-BOUND TAR RNA, NMR']",[(4S)-4-amino-4-carboxybutyl]-(diaminomethylid...,PDBBind_2,small_molecule
2,316d,3,1997,7.110138,0.544558,77.6nM,316d.pdf,PXF,IC50,GAAGCTTC,Cc1c2oc3c(C)cc(F)c(C(=O)O)c3nc-2c(C(=O)O)c(N)c1=O,['Selectivity of F8-actinomycin D for RNA:DNA ...,PXF,PDBBind_1,small_molecule


## Aptagen

In [ ]:
gen_df = pd.read_csv('../data/row/aptagen_small_molecules.csv', index_col=0)

In [ ]:
#Check the target category
gen_df[['ID', 'Target', 'Category', 'smiles']].head(5)


,ID,Target,Category,smiles
9,8118,Metronidazole,Small Organic,CC1=NC=C(N1CCO)[N+](=O)[O-]
17,7854,Ampicillin,Small Organic,CC1([C@@H](N2[C@H](S1)[C@@H](C2=O)NC(=O)[C@@H]...
18,7855,Ampicillin,Small Organic,CC1([C@@H](N2[C@H](S1)[C@@H](C2=O)NC(=O)[C@@H]...
19,8120,Cocaine,Small Organic,CN1[C@H]2CC[C@@H]1[C@H]([C@H](C2)OC(=O)C3=CC=C...
21,7834,Hemin,Protein,CC1=C(C2=CC3=C(C(=C([N-]3)C=C4C(=C(C(=N4)C=C5C...


In [11]:
gen_df.drop(columns=['ID'], inplace=True)
gen_df.rename(columns={'Sequence': 'apt_seq', 'Target': 'target_name', 'smiles': 'target_seq'}, inplace=True)
gen_df.drop_duplicates(subset=['apt_seq', 'target_seq', 'target_name'], inplace=True)


In [12]:
unique_sequences = gen_df['apt_seq'].drop_duplicates().reset_index(drop=True)
id_map = {
    seq: f"Aptagen_{i+1}" for i, seq in enumerate(unique_sequences)
}
gen_df['apt_name'] = gen_df['apt_seq'].map(id_map)


In [13]:
small_mol_targets = ['Hemin', 'Aflatoxin M1', 'adenosine', 'ATP', 'Kanamycin', 'Isocarbophos']

gen_df.loc[gen_df['target_name'].isin(small_mol_targets), 'Category'] = 'Small Organic'
gen_df = gen_df[~gen_df['Category'].isin(['Protein', 'Peptide'])]
gen_df['target_type'] = 'small_molecule'

## Published_ds

In [ ]:
pub_df = pd.read_csv('../data/row/aptamer_limited_ds.csv', index_col=0)

In [15]:
unique_sequences = pub_df['aptamer_sequence'].drop_duplicates().reset_index(drop=True)
id_map = {
    seq: f"Aptamer_ds_{i+1}" for i, seq in enumerate(unique_sequences)
}
pub_df['apt_name'] = pub_df['aptamer_sequence'].map(id_map)

pub_df.rename(columns={'aptamer_sequence': 'apt_seq', 'molecular_target_name': 'target_name', 'molecular_target_smiles': 'target_seq'}, inplace=True)
pub_df.drop(columns='aptamer_id', inplace=True)
pub_df

,apt_seq,target_name,target_seq,kd_value,apt_name
0,CCTGGGGGAGTATTGCGGAGGAAGG,Adenosine Triphosphate,C1=NC(=C2C(=N1)N(C=N2)C3C(C(C(O3)COP(=O)(O)OP(...,1.59e-07,Aptamer_ds_1
1,AAAGCGGGCGGTTGTATAGCGGAA,Adenosine Monophosphate,C1=NC(=C2C(=N1)N(C=N2)C3C(C(C(O3)COP(=O)(O)O)O...,6.33e-07,Aptamer_ds_2
2,GTCTCTGTGTGCGCCAGAGAACACTGGGGCAGATATGGGCCAGCAC...,Dopamine,C1=CC(=C(C=C1CCN)O)O,4e-08,Aptamer_ds_3
3,CTCAGTTCGGGACGACGGCAAGGTAACGTATGGGACCTTGGCACGA...,Thiamethoxam,CN1COCN(C1=N[N+](=O)[O-])CC2=CN=C(S2)Cl,54.63,Aptamer_ds_4
4,TAGGGAAGAGAAGGACATATGATCTGCGTTTATCTCCGCTCGTTAA...,Testosterone,CC12CCC3C(C1CCC2O)CCC4=CC(=O)CCC34C,10.7,Aptamer_ds_5
...,...,...,...,...,...
553,GACGACCATTGGTCTCTTTAGAGGTGTCCGTTAGGTCGTC,Theophylline,CN1C2=C(C(=O)N(C1=O)C)NC=N2,active,Aptamer_ds_443
554,GACGACGTTGTGGTCTATTCATAGGTGTCTGCATCGTCGTC,Theophylline,CN1C2=C(C(=O)N(C1=O)C)NC=N2,active,Aptamer_ds_444
555,GACGACGGTACAGGTCTATTCATAGGTGTCCGCAACGTCGTC,Theophylline,CN1C2=C(C(=O)N(C1=O)C)NC=N2,active,Aptamer_ds_445
556,GACGACCAGTGAGGTCTATTCATAGGTGTCCGCCAGGTCGTC,Theophylline,CN1C2=C(C(=O)N(C1=O)C)NC=N2,active,Aptamer_ds_446


In [16]:
pub_df.drop_duplicates(subset=['apt_seq', 'target_seq', 'target_name', 'apt_name'], inplace=True)
pub_df['target_type'] = 'small_molecule'

In [ ]:
merged_df = pd.concat([apipred_df, rsa_df, bind_df, gen_df, pub_df], ignore_index=True)
merged_df.to_csv('../data/row/new_aptamers_interactions.csv')

In [ ]:
def identify_sequence_type(seq):
    """Identify whether a nucleotide sequence is DNA or RNA."""
    seq = str(seq).upper()
    
    if 'U' in seq and 'T' in seq:
        return 'RNA'
    elif 'U' in seq:
        return 'RNA'
    elif 'T' in seq:
        return 'DNA'
    elif set(seq).issubset({'A', 'C', 'G'}):
        return 'DNA'
    else:
        return 'DNA'

In [ ]:
merged_df['type'] = merged_df['apt_seq'].apply(identify_sequence_type)
merged_df['subtype'] = 'aptamer'
merged_df['representation_type'] = 'sequence'
merged_df.rename(columns = {'apt_name': 'name', 'apt_seq':'content'}, inplace = True)

In [ ]:
aptamers_df = merged_df[['name', 'type', 'subtype', 'representation_type', 'content', 'URL']]
aptamers_df.drop_duplicates(subset=['content'], inplace=True)
aptamers_df.to_csv('../data/row/aptamers.csv')

In [ ]:
molecules_df = merged_df[['target_name', 'target_seq', 'target_type']]
molecules_df['representation_type'] = molecules_df['target_type'].map({
    'small_molecule': 'smiles',
    'protein': 'sequence'
})

In [ ]:
molecules_df.rename(columns = {'target_name': 'name', 'target_seq':'content', 'target_type':'type'}, inplace = True)
molecules_df.drop_duplicates(inplace=True)
molecules_df.to_csv('../data/row/molecules_from_aptamers.csv')